# OTE — media → Cloudflare R2 (Colab)

Optimizes your masters into web variants and uploads them to R2, ready for **otes.me**.

**Before running:** in Google Drive, arrange your media as one folder per discipline, files named as ids:
```
OTE/
  photography/  morning-kitchen.jpg  cove.mov  ...
  film/         static.mp4  ...
  sound/  writing/  design/
```
- **Folder name = discipline** (use the slugs in `lib/content.ts`: photography, film, sound, writing, design).
- **Filename (no extension) = the work id.** Keep it lowercase, no spaces (use `-`).

Run the cells top to bottom.

In [ ]:
# 1 · Mount your Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2 · Install tools (ffmpeg for video, Pillow + boto3 for images/upload)
!apt-get -qq install -y ffmpeg >/dev/null
!pip -q install pillow pillow-heif boto3
print('tools ready')

In [ ]:
#@title 3 · Config  { display-mode: "form" }
#@markdown Paste your R2 API token values (from the Cloudflare dashboard). These stay in this private notebook only.
R2_ACCOUNT_ID = ""  #@param {type:"string"}
R2_ACCESS_KEY_ID = ""  #@param {type:"string"}
R2_SECRET_ACCESS_KEY = ""  #@param {type:"string"}
R2_BUCKET = "otes-media"  #@param {type:"string"}
#@markdown Path in your mounted Drive that holds the discipline folders:
DRIVE_ROOT = "/content/drive/MyDrive/OTE"  #@param {type:"string"}
#@markdown Untick to do a dry run (optimize + list, no upload):
UPLOAD = True  #@param {type:"boolean"}
print('config set · root =', DRIVE_ROOT, '· upload =', UPLOAD)

In [ ]:
# 4 · Optimize + upload
import io, base64, subprocess, re
from pathlib import Path
from PIL import Image, ImageOps
import boto3
from botocore.config import Config
try:
    import pillow_heif; pillow_heif.register_heif_opener()
except Exception:
    pass

IMG_EXT = {'.jpg','.jpeg','.png','.webp','.tif','.tiff','.heic','.heif'}
VID_EXT = {'.mp4','.mov','.m4v','.webm','.avi'}
SIZES = {'wall': 720, 'full': 2200}

s3 = None
if UPLOAD:
    s3 = boto3.client(
        's3',
        endpoint_url=f'https://{R2_ACCOUNT_ID}.r2.cloudflarestorage.com',
        aws_access_key_id=R2_ACCESS_KEY_ID,
        aws_secret_access_key=R2_SECRET_ACCESS_KEY,
        region_name='auto',
        config=Config(signature_version='s3v4', s3={'addressing_style': 'path'}),
    )

def slug(s):
    s = re.sub(r'[^a-z0-9]+', '-', s.lower().strip()).strip('-')
    return s or 'x'

def put(key, body, ctype):
    size = f'{len(body)/1024:.0f}KB' if isinstance(body, (bytes, bytearray)) else ''
    print('   ', key, size)
    if s3:
        s3.put_object(Bucket=R2_BUCKET, Key=key, Body=body, ContentType=ctype,
                      CacheControl='public, max-age=31536000, immutable')

def webp(img, size, q):
    im = ImageOps.exif_transpose(img).convert('RGB')
    im.thumbnail((size, size))
    b = io.BytesIO(); im.save(b, 'WEBP', quality=q); return b.getvalue()

def do_image(disc, wid, path):
    img = Image.open(path)
    for v, sz in SIZES.items():
        put(f'{disc}/{wid}/{v}.webp', webp(img, sz, 72 if v == 'wall' else 82), 'image/webp')
    blur = webp(img, 24, 40)
    uri = ('data:image/webp;base64,' + base64.b64encode(blur).decode()).encode()
    put(f'{disc}/{wid}/blur.txt', uri, 'text/plain')

def do_video(disc, wid, path):
    tmp = Path('/content/_tmp'); tmp.mkdir(exist_ok=True)
    poster, clip = tmp / 'p.png', tmp / 'clip.mp4'
    subprocess.run(['ffmpeg','-y','-loglevel','error','-ss','1','-i',str(path),
                    '-frames:v','1',str(poster)], check=True)
    subprocess.run(['ffmpeg','-y','-loglevel','error','-i',str(path),'-an',
                    '-vf',"scale='min(1920,iw)':-2",'-c:v','libx264','-crf','24',
                    '-preset','veryfast','-movflags','+faststart',str(clip)], check=True)
    img = Image.open(poster)
    put(f'{disc}/{wid}/poster.webp', webp(img, SIZES['full'], 82), 'image/webp')
    put(f'{disc}/{wid}/wall.webp', webp(img, SIZES['wall'], 72), 'image/webp')
    put(f'{disc}/{wid}/clip.mp4', open(clip, 'rb').read(), 'video/mp4')

root = Path(DRIVE_ROOT)
assert root.exists(), f'Not found: {root}  — check DRIVE_ROOT'
n = 0
for d in sorted(p for p in root.iterdir() if p.is_dir()):
    disc = slug(d.name)
    for f in sorted(d.iterdir()):
        if f.is_dir() or f.name.startswith('.'):
            continue
        ext, wid = f.suffix.lower(), slug(f.stem)
        print(f'{disc}/{wid}')
        try:
            if ext in IMG_EXT: do_image(disc, wid, f)
            elif ext in VID_EXT: do_video(disc, wid, f)
            else: print('    skip', ext); continue
            n += 1
        except Exception as e:
            print('    ERROR', e)
print(f'\nDone: {n} work(s)', 'uploaded to R2 bucket ' + R2_BUCKET if UPLOAD else '(dry run — nothing uploaded)')

## After it finishes

1. In **Vercel → Settings → Environment Variables**, set `NEXT_PUBLIC_MEDIA_BASE` to your R2 public URL (`https://pub-….r2.dev`) for **Production**, then **Redeploy**.
2. Reload **otes.me** — the panels whose ids match your uploaded files now show real images; anything not yet uploaded stays as placeholder art.

**Reminder:** each file's id (its name) must match an `id` in `lib/content.ts` to attach to a specific titled work. Send the file list and I'll align `content.ts` for you.

⚠️ This notebook holds your R2 secret keys once filled in — keep it private, don't share or commit it.